In [1]:
import shap
from sklearn.model_selection import train_test_split

X, Y = shap.datasets.adult()

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size = 0.2, random_state = 1)
X_train, X_validation, Y_train, Y_validation = train_test_split(X_train, Y_train, test_size = 0.125, random_state = 1)

In [2]:
import pandas as pd

training_dataset = pd.concat([pd.Series(Y_train, index = X_train.index, name = 'Income > 50K', dtype = int), X_train],  axis = 1)
test_dataset = pd.concat([pd.Series(Y_test, index = X_test.index, name = 'Income > 50K', dtype = int), X_test],  axis = 1)
validation_dataset = pd.concat([pd.Series(Y_validation, index = X_validation.index, name = 'Income > 50K', dtype = int), X_validation],  axis = 1)

training_dataset.to_csv('train.csv', index = False, header = False)
validation_dataset.to_csv('validation.csv', index = False, header = False)

In [3]:
import sagemaker, boto3, os
bucket = sagemaker.Session().default_bucket()

boto3.Session().resource('s3').Bucket(bucket).Object(os.path.join('sagemaker-deployment-demo', 'data/train.csv')).upload_file('train.csv')
boto3.Session().resource('s3').Bucket(bucket).Object(os.path.join('sagemaker-deployment-demo', 'data/validation.csv')).upload_file('validation.csv')

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/pydantic/_internal/_fields.py:192: UserWarning: Field name "json" in "MonitoringDatasetFormat" shadows an attribute in parent "Base"
  warnings.warn(


[01/21/25 07:11:13] INFO     Found credentials from IAM Role:                                   ]8;id=969271;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=645549;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/botocore/credentials.py#1075\1075]8;;\
                             BaseNotebookInstanceEc2InstanceRole                                                   

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


                    INFO     Found credentials from IAM Role:                                   ]8;id=19426;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=108122;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/botocore/credentials.py#1075\1075]8;;\
                             BaseNotebookInstanceEc2InstanceRole                                                   

[01/21/25 07:11:14] INFO     Found credentials from IAM Role:                                   ]8;id=548389;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=750796;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/botocore/credentials.py#1075\1075]8;;\
                             BaseNotebookInstanceEc2InstanceRole                                                   

                    INFO     Found credentials from IAM Role:                                   ]8;id=698627;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=483301;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/botocore/credentials.py#1075\1075]8;;\
                             BaseNotebookInstanceEc2InstanceRole                                                   

In [4]:
xgb_model = sagemaker.estimator.Estimator(
    image_uri = sagemaker.image_uris.retrieve(framework = 'xgboost', region = 'ap-southeast-1', version = '1.7-1'),
    role = sagemaker.get_execution_role(), instance_count = 1, instance_type = 'ml.m4.xlarge')

xgb_model.set_hyperparameters(max_depth = 5, eta = 0.2, gamma = 4, min_child_weight = 6, subsample = 0.7, 
                              objective = 'binary:logistic', num_round = 1000)

training_input = sagemaker.session.TrainingInput('s3://{}/sagemaker-deployment-demo/data/train.csv'.format(bucket), content_type = 'csv')
validation_input = sagemaker.session.TrainingInput('s3://{}/sagemaker-deployment-demo/data/validation.csv'.format(bucket), content_type = 'csv')

xgb_model.fit({'train': training_input, 'validation': validation_input}, wait = True)

[01/21/25 07:11:15] INFO     Ignoring unnecessary instance type: None.                            ]8;id=130972;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/image_uris.py\image_uris.py]8;;\:]8;id=512770;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/image_uris.py#528\528]8;;\

                    INFO     Found credentials from IAM Role:                                   ]8;id=214042;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=492539;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/botocore/credentials.py#1075\1075]8;;\
                             BaseNotebookInstanceEc2InstanceRole                                                   

                    INFO     Found credentials from IAM Role:                                   ]8;id=400565;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=263440;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/botocore/credentials.py#1075\1075]8;;\
                             BaseNotebookInstanceEc2InstanceRole                                                   

                    INFO     Found credentials from IAM Role:                                   ]8;id=379741;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=614450;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/botocore/credentials.py#1075\1075]8;;\
                             BaseNotebookInstanceEc2InstanceRole                                                   

[01/21/25 07:11:16] INFO     SageMaker Python SDK will collect telemetry to help us better  ]8;id=569377;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=852726;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/telemetry/telemetry_logging.py#90\90]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#confi                        
                             guring-and-using-defaults-with-the-sagemaker-python-sdk.                              

                    INFO     Creating training-job with name:                                       ]8;id=288604;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/session.py\session.py]8;;\:]8;id=473759;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/session.py#1042\1042]8;;\
                             sagemaker-xgboost-2025-01-21-07-11-16-108                                             

2025-01-21 07:11:16 Starting - Starting the training job...
..25-01-21 07:11:44 Starting - Preparing the instances for training.
..25-01-21 07:12:10 Downloading - Downloading input data.
.....01-21 07:12:40 Downloading - Downloading the training image.
.[2025-01-21 07:13:52.394 ip-10-0-228-70.ap-southeast-1.compute.internal:7 INFO utils.py:28] RULE_JOB_STOP_SIGNAL_FILENAME: None
[2025-01-21 07:13:52.419 ip-10-0-228-70.ap-southeast-1.compute.internal:7 INFO profiler_config_parser.py:111] User has disabled profiler.
[2025-01-21:07:13:52:INFO] Imported framework sagemaker_xgboost_container.training
[2025-01-21:07:13:52:INFO] Failed to parse hyperparameter objective value binary:logistic to Json.
Returning the value itself
[2025-01-21:07:13:52:INFO] No GPUs detected (normal if no gpus installed)
[2025-01-21:07:13:52:INFO] Running XGBoost Sagemaker in algorithm mode
[2025-01-21:07:13:52:INFO] Determined 0 GPU(s) available on the instance.
[2025-01-21:07:13:52:INFO] Determined delimiter of C

In [5]:
xgb_predictor = xgb_model.deploy(initial_instance_count = 1, instance_type = 'ml.m4.xlarge')

[01/21/25 07:14:34] INFO     Creating model with name: sagemaker-xgboost-2025-01-21-07-14-34-538    ]8;id=966171;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/session.py\session.py]8;;\:]8;id=189319;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/session.py#4094\4094]8;;\

[01/21/25 07:14:35] INFO     Creating endpoint-config with name                                     ]8;id=99583;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/session.py\session.py]8;;\:]8;id=757856;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/session.py#5889\5889]8;;\
                             sagemaker-xgboost-2025-01-21-07-14-34-538                                             

                    INFO     Creating endpoint with name sagemaker-xgboost-2025-01-21-07-14-34-538  ]8;id=52082;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/session.py\session.py]8;;\:]8;id=928234;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/session.py#4711\4711]8;;\

-------!

In [7]:
client = boto3.client('sagemaker-runtime')
response = client.invoke_endpoint(EndpointName = 'sagemaker-xgboost-2025-01-21-07-14-34-538', 
                                 Body = '62.0,6,4.0,6,8,0,4,0,0.0,0.0,66.0,39', ContentType = 'text/csv')
print(response)
print('Prediction:', response['Body'].read().decode('utf8'))

{'ResponseMetadata': {'RequestId': '4ae5a520-97a6-4c29-a0f4-872a24d86651', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amzn-requestid': '4ae5a520-97a6-4c29-a0f4-872a24d86651', 'x-amzn-invoked-production-variant': 'AllTraffic', 'date': 'Tue, 21 Jan 2025 07:19:57 GMT', 'content-type': 'text/csv; charset=utf-8', 'content-length': '21', 'connection': 'keep-alive'}, 'RetryAttempts': 0}, 'ContentType': 'text/csv; charset=utf-8', 'InvokedProductionVariant': 'AllTraffic', 'Body': <botocore.response.StreamingBody object at 0x7fcc7e225900>}
Prediction: 0.031007694080471992



In [8]:
test_dataset.to_csv('test.csv', index = False, header = False)